# Data Collection — SpaceX REST API

**IBM Data Science Capstone — SpaceX Falcon 9**

Repository: [https://github.com/MCerros/IBM-Data-Science-Capstone-SpaceX](https://github.com/MCerros/IBM-Data-Science-Capstone-SpaceX)

Direct notebook URL after upload:  
[https://github.com/MCerros/IBM-Data-Science-Capstone-SpaceX/blob/main/01_SpaceX_API_Data_Collection.ipynb](https://github.com/MCerros/IBM-Data-Science-Capstone-SpaceX/blob/main/01_SpaceX_API_Data_Collection.ipynb)

**Data integrity note:** This notebook uses the project CSV files stored in the same repository.
No rows, metrics, charts, or model scores are manually invented.

## Objective

Demonstrate the SpaceX REST API collection workflow:
1. Request past launches.
2. Backfill rocket, launchpad, payload and core attributes from API IDs.
3. Keep Falcon 9 launches in the project period.
4. Flatten the nested response into a tabular dataset.
5. Use the saved project snapshot `dataset_part_1.csv` for reproducible downstream analysis.

In [1]:
from pathlib import Path
import pandas as pd
import requests

DATA_FILE = Path("dataset_part_1.csv")
SPACEX_API = "https://api.spacexdata.com/v4"
PAST_LAUNCHES_ENDPOINT = f"{SPACEX_API}/launches/past"

print("Primary API endpoint:", PAST_LAUNCHES_ENDPOINT)
print("Backfill endpoints: /v4/rockets, /v4/launchpads, /v4/payloads, /v4/cores")

Primary API endpoint: https://api.spacexdata.com/v4/launches/past
Backfill endpoints: /v4/rockets, /v4/launchpads, /v4/payloads, /v4/cores


In [2]:
def _get_json(url, timeout=15):
    response = requests.get(url, timeout=timeout)
    response.raise_for_status()
    return response.json()

def collect_spacex_api_data():
    launches = _get_json(PAST_LAUNCHES_ENDPOINT)
    rows = []

    for launch in launches:
        launch_date = pd.to_datetime(launch.get("date_utc"), errors="coerce")
        if pd.isna(launch_date):
            continue
        if launch_date.date() > pd.Timestamp("2020-11-13").date():
            continue

        rocket = _get_json(f"{SPACEX_API}/rockets/{launch['rocket']}")
        if rocket.get("name") != "Falcon 9":
            continue

        launchpad = _get_json(f"{SPACEX_API}/launchpads/{launch['launchpad']}")

        payload = {}
        if launch.get("payloads"):
            payload = _get_json(f"{SPACEX_API}/payloads/{launch['payloads'][0]}")

        core_info = launch.get("cores", [{}])[0]
        core = {}
        if core_info.get("core"):
            core = _get_json(f"{SPACEX_API}/cores/{core_info['core']}")

        rows.append({
            "Date": launch_date.date().isoformat(),
            "BoosterVersion": rocket.get("name"),
            "PayloadMass": payload.get("mass_kg"),
            "Orbit": payload.get("orbit"),
            "LaunchSite": launchpad.get("name"),
            "Outcome": f"{core_info.get('landing_success')} {core_info.get('landing_type')}",
            "Flights": core_info.get("flight"),
            "GridFins": core_info.get("gridfins"),
            "Reused": core_info.get("reused"),
            "Legs": core_info.get("legs"),
            "LandingPad": core_info.get("landpad"),
            "Block": core.get("block"),
            "ReusedCount": core.get("reuse_count"),
            "Serial": core.get("serial"),
            "Longitude": launchpad.get("longitude"),
            "Latitude": launchpad.get("latitude"),
        })

    data = pd.DataFrame(rows)
    data.insert(0, "FlightNumber", range(1, len(data) + 1))
    if "PayloadMass" in data and data["PayloadMass"].isna().any():
        data["PayloadMass"] = data["PayloadMass"].fillna(data["PayloadMass"].mean())
    return data

## Reproducible project snapshot

The project repository contains the saved API result `dataset_part_1.csv`.
The next cell attempts a live API validation. If network access is unavailable,
it transparently continues with the saved dataset rather than fabricating data.

In [3]:
try:
    live_api_data = collect_spacex_api_data()
    print("Live API validation succeeded.")
    print("Live rows:", len(live_api_data))
except Exception as exc:
    live_api_data = None
    print("Live API validation unavailable in this runtime:")
    print(type(exc).__name__, "-", str(exc)[:180])

data = pd.read_csv(DATA_FILE)
print("\nSaved project dataset shape:", data.shape)
data.head()

Live API validation unavailable in this runtime:
ConnectionError - HTTPSConnectionPool(host='api.spacexdata.com', port=443): Max retries exceeded with url: /v4/launches/past (Caused by NameResolutionError("HTTPSConnection(host='api.spacexdata.com'

Saved project dataset shape: (90, 17)


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857


In [4]:
print("Launch sites:")
print(data["LaunchSite"].value_counts())
print("\nProject date range:", data["Date"].min(), "to", data["Date"].max())
print("\nColumns:", list(data.columns))

Launch sites:
LaunchSite
CCAFS SLC 40    55
KSC LC 39A      22
VAFB SLC 4E     13
Name: count, dtype: int64

Project date range: 2010-06-04 to 2020-11-05

Columns: ['FlightNumber', 'Date', 'BoosterVersion', 'PayloadMass', 'Orbit', 'LaunchSite', 'Outcome', 'Flights', 'GridFins', 'Reused', 'Legs', 'LandingPad', 'Block', 'ReusedCount', 'Serial', 'Longitude', 'Latitude']
